# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook guides you step-by-step through loading, inspecting, and analyzing a dataset defined with a [Croissant](https://mlcommons.org/croissant/) schema using the `mlcroissant` library.

### Dataset Source
The FAIR² dataset describes predictors for adoption of indigenous and modern knowledge for rangeland management in Northern Kenya.

**Croissant schema URL:**  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Let's load dataset metadata using `mlcroissant.Dataset`. We'll print a summary description for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset Name: {meta.name}\n\nDescription: {meta.description}\n")

## 2. Data Overview
Examine available record sets and their schemas. All entities (record sets, fields, columns) are referenced by their `@id` fields.

In [ ]:
# List all record sets available in the dataset
record_sets_info = dataset.schema.get('recordSet', [])

if not record_sets_info:
    print("No record sets found in the schema metadata.")
else:
    print("Record Sets in Dataset:")
    for rs in record_sets_info:
        print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")

Let's print the first record from each record set to see the available fields and their `@id`s.

In [ ]:
# Helper: Print records and show field @ids for each record set
record_sets_info = dataset.schema.get('recordSet', [])
example_records = {}

for rs in record_sets_info:
    rs_id = rs['@id']
    print(f"\n---\nRecord set: {rs.get('name', 'N/A')} (@id: {rs_id})")
    rs_fields = rs.get('field', [])
    if isinstance(rs_fields, dict):
        rs_fields = [rs_fields]
    print("Fields:")
    for f in rs_fields:
        if isinstance(f, dict):
            print(f"  - {f['@id']} (name: {f.get('name', '')})")
        else:
            print(f"  - {f}")
    print("Sample record:")
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            example_records[rs_id] = recs[0]
            print(recs[0])
        else:
            print('  [No data.]')
    except Exception as e:
        print(f"  [Error loading records: {e}]")

## 3. Data Extraction
Let's load data for all record sets into DataFrames for further analysis.
Each DataFrame uses the record set `@id` as key in a dictionary.

In [ ]:
# Get the list of record set @ids
record_sets_info = dataset.schema.get('recordSet', [])
all_recordset_ids = [rs['@id'] for rs in record_sets_info]
dataframes = {}

for rs_id in all_recordset_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows for record set '@id': {rs_id}")
        else:
            print(f"No records found for record set '@id': {rs_id}")
    except Exception as e:
        print(f"Error loading records for record set '@id': {rs_id}: {e}")

# If any record sets loaded, show available columns for the first one
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set '@id': {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes available.")

## 4. Exploratory Data Analysis (EDA)
We perform some typical exploratory operations such as filtering, normalization, and grouping on a numeric field using field `@id`s.
Please check above for a field that is numeric (e.g., 'log_likelihood', 'coefficient', etc), and a field to group by (e.g., 'ward', 'gender', etc).

In [ ]:
# <-- Modify these according to the printed field @ids above, e.g.:
# record_set_id = 'cr:OrderedLogitRegressionResults' (example)
# numeric_field_id = 'cr:coefficient' (example)
# group_field_id = 'cr:ward' (example)

# Fill these variables with the actual @id from earlier outputs
record_set_id = None
numeric_field_id = None
group_field_id = None

# Demo code assumes you've assigned these above (by copying from the printed @ids)
if (record_set_id is not None and 
    numeric_field_id is not None and 
    record_set_id in dataframes and 
    numeric_field_id in dataframes[record_set_id].columns):
    df = dataframes[record_set_id]
    
    # Filter for numeric_field > threshold
    threshold = 0  # Set as appropriate
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the field
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) /
        filtered_df[numeric_field_id].astype(float).std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
    
    # Optionally group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped statistics by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Please set the variables 'record_set_id', 'numeric_field_id', and 'group_field_id' above, based on your schema/fields.")

## 5. Visualization
Let's visualize the selected numeric field (for example: histogram or boxplot) and analyze its distribution across a group if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure variables are set per your data's context (see previous cell)
if (record_set_id and numeric_field_id and record_set_id in dataframes and numeric_field_id in dataframes[record_set_id].columns):
    df = dataframes[record_set_id]
    values = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(values, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id is set, make a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Please ensure the analysis variables ('record_set_id', 'numeric_field_id', optionally 'group_field_id') are correctly specified.")

## 6. Conclusion
In this notebook, you explored the FAIR² dataset using the Croissant schema and `mlcroissant` tools. You loaded metadata, inspected record sets and fields (referenced by their `@id`), loaded data into DataFrames, performed EDA, and visualized results.

You can build upon this notebook to conduct deeper statistical analyses or apply machine learning by continuing to use the `@id` references for consistent and reproducible data access.